<a href="https://colab.research.google.com/github/Reginajose/Observatorio-Clima-Grande-desafio/blob/main/C%C3%B3pia_de_Explora%C3%A7%C3%A3o_Atlas_S2iD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


##Diagnóstico de Qualidade de Dados: Desastres Hidrológicos no S2iD (2021–2025)

**Objetivo Metodológico:** Este notebook realiza uma **auditoria estática e não destrutiva** sobre a base de dados do Atlas Digital de Desastres (S2iD). O propósito aqui **não é apagar ou alterar precipitadamente os dados brutos**, mas sim "tirar uma radiografia" completa do dataset para identificar anomalias, criar etiquetas de auditoria (*flags* booleanas e colunas calculadas) e fundamentar a seção de qualidade da documentação técnica e do nosso **Data Card**.

---

### Fundamentação Teórica e Referência Metodológica

O método de diagnóstico e a avaliação por dimensões aplicadas neste código foram estruturadas com base nos princípios de qualidade de dados consolidados na revisão sistemática da literatura de **Lemos, Coelho Júnior e Martins (2023)**, integrando preceitos clássicos de governança e auditoria de bancos de dados (*Chapman, 2005; Eckerson, 2002; Batini &amp; Scannapieca, 2006*):

&gt; **Referência Bibliográfica de Suporte:**  
&gt; LEMOS, Daniela Lucas da Silva; COELHO JÚNIOR, Abeil; MARTINS, Dalton Lopes. **Modelos de diagnóstico de qualidade de dados no domínio do Patrimônio Cultural: uma revisão sistemática de literatura**. *Perspectivas em Ciência da Informação*, v. 28, e46064, 2023. DOI: [https://doi.org/10.1590/1981-5344/46064](https://doi.org/10.1590/1981-5344/46064).

---

### Escopo e Dimensões de Diagnóstico Avaliadas

Carregamos a base bruta do S2iD, aplicamos o filtro do projeto (**Brasil, 2021 a 2025, Desastres Hidrológicos - totalizando 9.890 ocorrências**) e submetemos o dataset às **6 dimensões essenciais de qualidade**:

1. **Completude:** Avaliação de nulos físicos (sintáticos) e sinalização de "falsos zeros" em prejuízos financeiros (semântica).
2. **Consistência Interna:** Validação da coerência matemática das subcategorias humanas e regras de status legal.
3. **Unicidade e Duplicidade:** Rastreamento de duplicatas por protocolo de sistema e por chave semântica (`IBGE + Data + Tipologia`).
4. **Acurácia e Outliers:** Mapeamento estatístico de valores extremos (\\(3 \times IQR\\)) para isolar catástrofes históricas atípicas de prejuízo típico.
5. **Atualidade e Cobertura Temporal:** Validação da conversão de datas e continuidade da série temporal de 5 anos.
6. **Conformidade de Formato:** Validação por Expressões Regulares (*Regex*) de códigos IBGE (7 dígitos) e siglas de UF.

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
# Importamos a biblioteca 'pandas', que é a ferramenta padrão do Python para ler e analisar tabelas
import pandas as pd

# 1. Carregamento da base de dados
# Indicamos o nome exato do arquivo. O 'sep=";"' avisa ao computador que as colunas no Brasil são separadas por ponto e vírgula
caminho_arquivo = '/content/drive/MyDrive/atlas_geral.csv'

print("Lendo o arquivo original...")
df_bruto = pd.read_csv(caminho_arquivo, sep=';', encoding='utf-8')

print(f"Base carregada com sucesso! O arquivo inteiro tem {df_bruto.shape[0]} linhas e {df_bruto.shape[1]} colunas.")

Lendo o arquivo original...
Base carregada com sucesso! O arquivo inteiro tem 76190 linhas e 70 colunas.


/tmp/ipykernel_5003/1326060330.py:9: DtypeWarning: Columns (11,43,44,45,46,47,48,49,62) have mixed types. Specify dtype option on import or set low_memory=False.
  df_bruto = pd.read_csv(caminho_arquivo, sep=';', encoding='utf-8')


## 1. Aplicando o Recorte do Projeto
A base contém o Brasil inteiro desde 1991. Para avaliar a qualidade do dado que *nós* vamos usar, primeiro precisamos isolar apenas o nosso alvo: estados do Sul (RS, SC, PR), anos de 2021 a 2025, e o grupo Hidrológico (que engloba Inundações, Enxurradas, Alagamentos, Chuvas Intensas e Movimento de Massa).

In [20]:
df_bruto['Data_Evento_dt'] = pd.to_datetime(df_bruto['Data_Evento'], format='%d/%m/%Y', errors='coerce')

# List of all Brazilian state acronyms
estados_brasil = [
    'AC', 'AL', 'AP', 'AM', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MT', 'MS', 'MG',
    'PA', 'PB', 'PR', 'PE', 'PI', 'RJ', 'RN', 'RS', 'RO', 'RR', 'SC', 'SP', 'SE', 'TO'
]
filtro_estado = df_bruto['Sigla_UF'].isin(estados_brasil)
filtro_ano = (df_bruto['Data_Evento_dt'].dt.year >= 2021) & (df_bruto['Data_Evento_dt'].dt.year <= 2025)
filtro_grupo = df_bruto['grupo_de_desastre'] == 'Hidrológico'

df_recorte = df_bruto[filtro_estado & filtro_ano & filtro_grupo].copy()

print(f"Pronto! Nosso recorte tem exatamente {len(df_recorte)} registros de desastres.")

Pronto! Nosso recorte tem exatamente 9890 registros de desastres.


## 2. Estado Atual da Base (Distribuição Geográfica e Tipologias)
Agora vamos pedir para o computador contar quantas linhas existem para cada estado, quais os tipos exatos de desastres que sobraram no recorte e qual o status de reconhecimento deles no sistema S2iD.

In [21]:
# O comando 'value_counts()' agrupa os itens iguais e conta quantos existem de cada
print("--- DISTRIBUIÇÃO POR ESTADO ---")
print(df_recorte['Sigla_UF'].value_counts())

print("\n--- QUAIS TIPOS DE DESASTRES EXISTEM NESTE RECORTE? ---")
print(df_recorte['descricao_tipologia'].value_counts())

print("\n--- STATUS LEGAL DOS DESASTRES ---")
# Mostra se o desastre foi apenas registrado pela prefeitura ou se já foi reconhecido oficialmente pelo Governo
print(df_recorte['Status'].value_counts())

--- DISTRIBUIÇÃO POR ESTADO ---
Sigla_UF
SC    1989
MG    1759
RS    1517
PA     598
BA     554
PE     389
RJ     377
ES     322
MS     288
SP     259
MA     252
PR     250
AM     217
TO     161
MT     153
RN     152
AL     146
CE     123
GO     106
AC      81
SE      51
RO      41
PI      40
RR      26
AP      25
PB      12
DF       2
Name: count, dtype: int64

--- QUAIS TIPOS DE DESASTRES EXISTEM NESTE RECORTE? ---
descricao_tipologia
Chuvas Intensas       6507
Inundações            1010
Enxurradas             909
Alagamentos            819
Movimento de Massa     645
Name: count, dtype: int64

--- STATUS LEGAL DOS DESASTRES ---
Status
Registro       5255
Reconhecido    4635
Name: count, dtype: int64



##3. Diagnóstico de Qualidade: Completude (Completeness)

O que foi avaliado: a presença física de dados (nulos) e a validade semântica dos valores preenchidos.

**Completude Sintática (Nulos Físicos)**: o resultado mostrou 0% de campos nulos nas colunas vitais.

**Completude Semântica ("Falsos Zeros")**: o resultado mostrou que 36,1% dos registros (3.567 ocorrências) possuem valor R$ 0,00 simultaneamente em Danos Materiais e Prejuízos Públicos. A interpretação é que isso não indica ausência de estrago real, mas sim o registro de emergência efetuado antes da conclusão do laudo pericial de prejuízos pelo município. Como tratamento, foi gerada a coluna sinalizadora flag_sem_declaracao_financeira (booleana) para isolar esses 36,1% em análises de custo e evitar a subestimação da média de prejuízos.

In [22]:
import pandas as pd
import numpy as np

def diagnostico_completude(df, colunas_vitais):
    total_linhas = len(df)
    # 1. Completude Sintática (Nulos Físicos)
    nulos_abs = df[colunas_vitais].isnull().sum()
    nulos_pct = (nulos_abs / total_linhas) * 100

    # 2. Completude Semântica ("Falsos Zeros" Financeiros)
    zerados_mas_com_status = df[
        (df['DM_total_danos_materiais'] == 0) &
        (df['PEPL_total_publico'] == 0)
    ]
    pct_falsos_zeros = (len(zerados_mas_com_status) / total_linhas) * 100

    # Adicionando a flag ao DataFrame
    df['flag_sem_declaracao_financeira'] = ((df['DM_total_danos_materiais'] == 0) &
                                            (df['PEPL_total_publico'] == 0))

    # Relatório Sintético
    df_completude = pd.DataFrame({
        'Nulos_Sintaticos_Abs': nulos_abs,
        'Nulos_Sintaticos_Pct (%)': nulos_pct.round(2)
    })

    print("=== DIAGNÓSTICO DE COMPLETUDE ===")
    print(df_completude)
    print(f"\n[Alerta Semântico] Registros com R$ 0,00 em Danos Materiais e Públicos: "
          f"{len(zerados_mas_com_status)} ocorrências ({pct_falsos_zeros:.2f}% do dataset).")
    return df, df_completude, zerados_mas_com_status

# Execução no recorte do Colab
colunas_vitais = [
    'Cod_IBGE_Mun',
    'Nome_Municipio',
    'Sigla_UF',
    'Data_Evento',
    'Status',
    'descricao_tipologia',
    'Protocolo_S2iD',
    'DM_total_danos_materiais',
    'PEPL_total_publico',
    'DH_MORTOS',
    'DH_DESALOJADOS',
    'DH_total_danos_humanos_diretos'
]
df_recorte, res_completude, df_falsos_zeros = diagnostico_completude(df_recorte, colunas_vitais)

=== DIAGNÓSTICO DE COMPLETUDE ===
                                Nulos_Sintaticos_Abs  Nulos_Sintaticos_Pct (%)
Cod_IBGE_Mun                                       0                       0.0
Nome_Municipio                                     0                       0.0
Sigla_UF                                           0                       0.0
Data_Evento                                        0                       0.0
Status                                             0                       0.0
descricao_tipologia                                0                       0.0
Protocolo_S2iD                                     0                       0.0
DM_total_danos_materiais                           0                       0.0
PEPL_total_publico                                 0                       0.0
DH_MORTOS                                          0                       0.0
DH_DESALOJADOS                                     0                       0.0
DH_total_danos_hum

##2. Diagnóstico de Qualidade: Consistência Interna

*O que foi avaliado:* A coerência matemática e lógica entre diferentes colunas do mesmo registro, validando o cumprimento das regras de negócio do S2iD.

🛠️ **Entendendo as Novas Colunas Criadas**

 Para garantir um dataset auditável e reproduzível sem excluir dados brutos, foram incorporadas **3 novas colunas** ao dataset. Abaixo explicamos a função de cada uma para orientar a equipe de pesquisa:

 1. 'DH\_total\_calculado' *(Tipo: Numérico)*

 O que é: O resultado exato da soma matemática realizada via código: \`DH\_MORTOS\` + \`DH\_DESALOJADOS\` + \`DH\_DESABRIGADOS\` + \`DH\_ENFERMOS\` + \`DH\_DESAPARECIDOS\`.

 **Por que foi criada:** Em 6,13% dos casos, o total oficial da base original continha erros de digitação/consolidação municipal. Esta coluna garante precisão matemática auditada.
 **Como usar na prática:** Utilize \`DH\_total\_calculado\` como a métrica principal para qualquer cálculo, estatística descritiva ou gráfico sobre o total de vítimas humanas.

 2. 'flag\_inconsistencia\_soma\_humana\' *(Tipo: Booleano - True/False)*

 **O que é:** Etiqueta que marca \`True\` exatamente nas **606 linhas (6,13%)** onde a conta original não fechou, e \`False\` onde a conta estava perfeita.

 **Por que foi criada:** Dá transparência ao erro sem deletar informações. Permite rastrear exatamente quais registros apresentavam desasincronia nos cadastros municipais.
 **Como usar na prática:** Caso a pesquisa exija um filtro ultra-rigoroso focado apenas em registros sem divergências cadastrais, basta filtrar a base por \`df[df['flag\_inconsistencia\_soma\_humana'] == False]\`.

 3. 'flag\_reconhecido\_sem\_impacto\' *(Tipo: Booleano - True/False)*

 **O que é:** Etiqueta que marca \`True\` nas **177 linhas (1,79%)** de desastres homologados pelo Governo Federal, mas que não possuem nenhum dano humano ou financeiro cadastrado no sistema.

 **Por que foi criada:** Separa a dimensão \*\*administrativa/legal\*\* (o reconhecimento da emergência) da dimensão \*\*estatística de impacto

 **Como usar na prática:** Ao calcular o prejuízo financeiro médio ou a média de afetados por evento reconhecido, recomenda-se filtrar esses 177 casos (\`flag\_reconhecido\_sem\_impacto == False\`) para evitar distorções nas médias.

In [23]:
import pandas as pd
import numpy as np

def diagnostico_e_flags_consistencia(df):
    total_linhas = len(df)

    # 1. Calculando a soma real das subcategorias humanas
    soma_humanos_calculada = (
        df['DH_MORTOS'].fillna(0) +
        df['DH_DESALOJADOS'].fillna(0) +
        df['DH_DESABRIGADOS'].fillna(0) +
        df['DH_ENFERMOS'].fillna(0) +
        df['DH_DESAPARECIDOS'].fillna(0)
    )

    # Criando a Coluna Derivada Auditada
    df['DH_total_calculado'] = soma_humanos_calculada

    # FLAG 1: Sinaliza se a soma não bate com o total oficial
    df['flag_inconsistencia_soma_humana'] = (soma_humanos_calculada != df['DH_total_danos_humanos_diretos'].fillna(0))

    # FLAG 2: Sinaliza se foi reconhecido mas não tem nenhum impacto
    sem_impacto_algum = (
        (df['DM_total_danos_materiais'] == 0) &
        (df['PEPL_total_publico'] == 0) &
        (df['DH_total_danos_humanos_diretos'] == 0)
    )
    df['flag_reconhecido_sem_impacto'] = (df['Status'] == 'Reconhecido') & sem_impacto_algum

    # Quantificação
    qtd_inc_soma = df['flag_inconsistencia_soma_humana'].sum()
    qtd_inc_status = df['flag_reconhecido_sem_impacto'].sum()

    relatorio = pd.DataFrame({
        'Regra_Consistencia': [
            'Soma de Danos Humanos Diretos == Total Oficial',
            'Desastre Reconhecido sem Impactos Declarados'
        ],
        'Inconsistencias_Detectadas': [qtd_inc_soma, qtd_inc_status],
        'Impacto_Percentual (%)': [
            round((qtd_inc_soma / total_linhas) * 100, 2),
            round((qtd_inc_status / total_linhas) * 100, 2)
        ]
    })

    print("=== DIAGNÓSTICO E AUDITORIA DE CONSISTÊNCIA INTERNA ===")
    print(relatorio.to_string(index=False))
    print("\n[Colunas Criadas no Dataset]:")
    print(" - 'DH_total_calculado': Soma auditada das subcategorias humanas.")
    print(" - 'flag_inconsistencia_soma_humana': True para os 606 registros com divergência.")
    print(" - 'flag_reconhecido_sem_impacto': True para os 177 registros sem danos declarados.")
    return df, relatorio

# Execução no seu recorte
df_recorte, res_consistencia = diagnostico_e_flags_consistencia(df_recorte)


=== DIAGNÓSTICO E AUDITORIA DE CONSISTÊNCIA INTERNA ===
                            Regra_Consistencia  Inconsistencias_Detectadas  Impacto_Percentual (%)
Soma de Danos Humanos Diretos == Total Oficial                         606                    6.13
  Desastre Reconhecido sem Impactos Declarados                         177                    1.79

[Colunas Criadas no Dataset]:
 - 'DH_total_calculado': Soma auditada das subcategorias humanas.
 - 'flag_inconsistencia_soma_humana': True para os 606 registros com divergência.
 - 'flag_reconhecido_sem_impacto': True para os 177 registros sem danos declarados.


## 3\. Diagnóstico de Qualidade: Unicidade e Duplicidade

**🛠️ Decisão Metodológica e Tratamento**

**Baixa Incidência:** A duplicidade semântica afeta apenas 0,55% do dataset (54 registros em 9.890), confirmando a alta integridade cadastral da base.

**Tratamento Não Destrutivo:** Em vez de excluir diretamente esses registros, a coluna booleana `flag_duplicata_semantica` foi mantida como etiqueta. Para análises de somatória de prejuízos e contagem de eventos, a recomendação é consolidar essas duplicatas mantendo o registro de maior `Status` ou de maior valor de dano informado.




In [24]:
def diagnostico_unicidade(df):
    total_linhas = len(df)

    # 1. Duplicidade por Chave Primária (Protocolo S2iD)
    # keep=False marca todas as duplicatas (incluindo a primeira ocorrência)
    df['flag_duplicata_protocolo'] = df.duplicated(subset=['Protocolo_S2iD'], keep=False)
    dup_protocolo = df['flag_duplicata_protocolo'].sum()

    # 2. Duplicidade Semântica (Mesmo Município + Mesma Data + Mesma Tipologia)
    chave_semantica = ['Cod_IBGE_Mun', 'Data_Evento', 'descricao_tipologia']
    df['flag_duplicata_semantica'] = df.duplicated(subset=chave_semantica, keep=False)
    dup_semanticos = df['flag_duplicata_semantica'].sum()

    relatorio_unicidade = pd.DataFrame({
        'Criterio_Unicidade': ['Chave Primária (Protocolo_S2iD)', 'Chave Semântica (IBGE + Data + Tipologia)'],
        'Registros_Duplicados': [dup_protocolo, dup_semanticos],
        'Percentual_Afetado (%)': [
            (dup_protocolo / total_linhas) * 100,
            (dup_semanticos / total_linhas) * 100
        ]
    })

    print("=== DIAGNÓSTICO DE UNICIDADE ===")
    print(relatorio_unicidade.to_string(index=False))
    return df, relatorio_unicidade

df_recorte, res_unicidade = diagnostico_unicidade(df_recorte)

=== DIAGNÓSTICO DE UNICIDADE ===
                       Criterio_Unicidade  Registros_Duplicados  Percentual_Afetado (%)
          Chave Primária (Protocolo_S2iD)                     0                0.000000
Chave Semântica (IBGE + Data + Tipologia)                    54                0.546006


##4.Diagnóstico de Qualidade: Acurácia, Validade e Outliers

**O que foi avaliado:** A plausibilidade numérica dos dados e a identificação de discrepâncias extremas (outliers severos) em prejuízos financeiros e danos humanos, utilizando o critério estatístico da amplitude interquartil severa (\\\\(Q3 + 3 \\times IQR\\\\)).

**Top 3 Maiores Eventos em Prejuízo Material Registrados**
 1. Maceió / AL (16/03/2021) - Movimento de Massa: **R$$ 7.651.464.416**(Crise de afundamento de solo urbanizado)

 2. São Leopoldo / RS (27/04/2024) - Chuvas Intensas: **R$4.823.240.240,88** (Enchentes históricas do Rio Grande do Sul).

 3. Pedro Leopoldo / MG (09/01/2022) - Chuvas Intensas: **R$ 1.184.366.614,71** (Inundações severas do início de 2022 em MG)

### Decisão Metodológica e Novas Colunas

Para viabilizar tanto a análise do custo total do país quanto o estudo de padrões típicos de eventos municipais recorrentes, foram incorporadas duas sinalizações booleanas:

1. **`flag_outlier_danos_materiais` (946 registros / 9,57%):**
   - Etiqueta `True` em desastres com prejuízos superiores a R\$ 5,31 milhões.
   - **Aplicação prática:** Permite calcular a **mediana** ou filtrar a base (`flag_outlier_danos_materiais == False`) para estimar o prejuízo médio típico sem a distorção dos mega-eventos de bilhões de reais.

2. **`flag_outlier_humanos` (1.456 registros / 14,72%):**
   - Etiqueta `True` em ocorrências com mais de 280 pessoas afetadas.
   - **Aplicação prática:** Permite separar eventos locais pequenos de desastres socioambientais de escala regional.

```

In [25]:
import pandas as pd
import numpy as np

def diagnostico_e_flags_outliers(df):
    total_linhas = len(df)

    # 1. Análise de Outliers Severos em Danos Materiais (Regra do 3 * IQR)
    dm = df['DM_total_danos_materiais'].dropna()
    q1_dm = dm.quantile(0.25)
    q3_dm = dm.quantile(0.75)
    iqr_dm = q3_dm - q1_dm
    limite_sup_dm = q3_dm + (3.0 * iqr_dm)

    # Criando a FLAG para Danos Materiais Extremos
    df['flag_outlier_danos_materiais'] = df['DM_total_danos_materiais'] > limite_sup_dm
    qtd_outliers_dm = df['flag_outlier_danos_materiais'].sum()

    # 2. Análise de Outliers Severos em Danos Humanos Totais (Regra do 3 * IQR)
    dh = df['DH_total_calculado'].dropna()
    q1_dh = dh.quantile(0.25)
    q3_dh = dh.quantile(0.75)
    iqr_dh = q3_dh - q1_dh
    limite_sup_dh = q3_dh + (3.0 * iqr_dh)

    # Criando a FLAG para Afetados Humanos Extremos
    df['flag_outlier_humanos'] = df['DH_total_calculado'] > limite_sup_dh
    qtd_outliers_dh = df['flag_outlier_humanos'].sum()

    # Tabela Resumo
    relatorio_outliers = pd.DataFrame({
        'Métrica_Analisada': ['Danos Materiais (R$)', 'Total de Afetados Humanos (Pessoas)'],
        'Corte_Estatístico_Severo (3*IQR)': [f"R$ {limite_sup_dm:,.2f}", f"{limite_sup_dh:,.0f} pessoas"],
        'Qtd_Outliers_Detectados': [qtd_outliers_dm, qtd_outliers_dh],
        'Impacto_Percentual (%)': [
            round((qtd_outliers_dm / total_linhas) * 100, 2),
            round((qtd_outliers_dh / total_linhas) * 100, 2)
        ]
    })

    print("=== DIAGNÓSTICO DE ACURÁCIA E OUTLIERS ===")
    print(relatorio_outliers.to_string(index=False))

    print("\n[Top 3 Maiores Eventos em Prejuízo Material]:")
    top3_dm = df.sort_values(by='DM_total_danos_materiais', ascending=False)[
        ['Nome_Municipio', 'Sigla_UF', 'Data_Evento', 'descricao_tipologia', 'DM_total_danos_materiais']
    ].head(3)
    print(top3_dm.to_string(index=False))

    return df, relatorio_outliers

# Execução no seu recorte
df_recorte, res_outliers = diagnostico_e_flags_outliers(df_recorte)


=== DIAGNÓSTICO DE ACURÁCIA E OUTLIERS ===
                  Métrica_Analisada Corte_Estatístico_Severo (3*IQR)  Qtd_Outliers_Detectados  Impacto_Percentual (%)
               Danos Materiais (R$)                  R$ 5,309,228.56                      946                    9.57
Total de Afetados Humanos (Pessoas)                      280 pessoas                     1456                   14.72

[Top 3 Maiores Eventos em Prejuízo Material]:
Nome_Municipio Sigla_UF Data_Evento descricao_tipologia  DM_total_danos_materiais
        Maceió       AL  16/03/2021  Movimento de Massa              7.651464e+09
  São Leopoldo       RS  27/04/2024     Chuvas Intensas              4.823240e+09
Pedro Leopoldo       MG  09/01/2022     Chuvas Intensas              1.184367e+09


## 5. Diagnóstico de Qualidade: Atualidade e Cobertura Temporal

**O que foi avaliado:** A integridade da janela temporal da pesquisa (2021 a 2025), a ausência de erros na conversão de datas e a distribuição anual das ocorrências de desastres hidrológicos.

📅 Distribuição das Ocorrências por Ano de Registro

| Ano do Evento | Quantidade de Ocorrências | Proporção no Dataset (%) | Destaque Histórico do Período |
| :---: | :---: | :---: | :--- |
| **2021** | 1.334 | 13,49% | Início do recorte da pesquisa. |
| **2022** | 2.777 | 28,08% | Ano com maior volume de decretações hidrológicas no país. |
| **2023** | 2.361 | 23,87% | Manutenção de alto volume de eventos pluviométricos. |
| **2024** | 1.739 | 17,58% | Eventos de grande escala no Sul do país. |
| **2025** | 1.679 | 16,98% | Fechamento do período de amostragem. |
| **TOTAL** | **9.890** | **100,00%** | **Série temporal contínua e sem lacunas.** |

#🛠️ Decisão Metodológica

**Validação de Calendário:** A ausência de falhas na parsing de datas confirma que a ordenação cronológica e as análises sazonais (mensais e anuais) podem ser conduzidas sem perda de registros por erro de formatação.

In [26]:
def diagnostico_atualidade(df, coluna_data):
    total_linhas = len(df)
    # Converte a coluna de data para datetime, forçando erros para NaN
    datas_dt = pd.to_datetime(df[coluna_data], format='%d/%m/%Y', errors='coerce')

    # 1. Datas Inválidas/Incompatíveis (que não puderam ser convertidas)
    datas_invalidas = datas_dt.isnull().sum()

    # 2. Datas Fora do Escopo (2021-2025)
    # Usamos o `dt.year` para extrair o ano e verificar se está fora do intervalo
    fora_do_escopo = ((datas_dt.dt.year < 2021) | (datas_dt.dt.year > 2025)).sum()

    # 3. Distribuição Anual de Ocorrências
    distribuicao_anual = datas_dt.dt.year.value_counts().sort_index().to_dict()

    relatorio = pd.DataFrame({
        'Métrica_Temporal': ['Datas Inválidas/Incompatíveis', 'Datas Fora do Escopo (2021-2025)'],
        'Quantidade': [datas_invalidas, fora_do_escopo],
        'Percentual (%)': [
            (datas_invalidas / total_linhas) * 100,
            (fora_do_escopo / total_linhas) * 100
        ]
    })

    print("=== DIAGNÓSTICO DE ATUALIDADE E COBERTURA TEMPORAL ===")
    print(relatorio.to_string(index=False))
    print(f"\nDistribuição de Ocorrências por Ano: {distribuicao_anual}")
    return relatorio

res_atualidade = diagnostico_atualidade(df_recorte, 'Data_Evento')

=== DIAGNÓSTICO DE ATUALIDADE E COBERTURA TEMPORAL ===
                Métrica_Temporal  Quantidade  Percentual (%)
   Datas Inválidas/Incompatíveis           0             0.0
Datas Fora do Escopo (2021-2025)           0             0.0

Distribuição de Ocorrências por Ano: {2021: 1334, 2022: 2777, 2023: 2361, 2024: 1739, 2025: 1679}


##6. Diagnóstico de Qualidade: Conformidade de Formato e Padronização
**O que foi avaliado:** Aderência estrita dos campos de identificação espacial e administrativa aos padrões oficiais (Expressões Regulares / Regex), verificando a estrutura do Código IBGE de 7 dígitos, siglas de UF e ausência de caracteres/espaços invisíveis.

In [27]:
import re

def diagnostico_conformidade_formato(df):
    total_linhas = len(df)

    # 1. Código IBGE (Exatamente 7 dígitos numéricos)
    padrao_ibge = re.compile(r'^\d{7}$', re.IGNORECASE)
    # Primeiro, converte para string para lidar com possíveis nulos ou tipos mistos
    ibge_inconformes = ~df['Cod_IBGE_Mun'].astype(str).apply(lambda x: bool(padrao_ibge.match(x)))

    # 2. Espaços invisíveis em Municípios (nas pontas)
    nomes_originais = df['Nome_Municipio'].astype(str)
    espacos_extras = (nomes_originais != nomes_originais.str.strip())

    # 3. Sigla UF (2 letras maiúsculas)
    padrao_uf = re.compile(r'^[A-Z]{2}$', re.IGNORECASE)
    uf_inconformes = ~df['Sigla_UF'].astype(str).apply(lambda x: bool(padrao_uf.match(x)))

    relatorio_formato = pd.DataFrame({
        'Regra_Formato': [
            'Cod_IBGE_Mun (7 dígitos numéricos)',
            'Nome_Municipio (sem espaços extras nas pontas)',
            'Sigla_UF (2 letras maiúsculas)'
        ],
        'Inconformidades': [ibge_inconformes.sum(), espacos_extras.sum(), uf_inconformes.sum()],
        'Percentual_Afetado (%)': [
            (ibge_inconformes.sum() / total_linhas) * 100,
            (espacos_extras.sum() / total_linhas) * 100,
            (uf_inconformes.sum() / total_linhas) * 100
        ]
    })

    print("=== DIAGNÓSTICO DE CONFORMIDADE DE FORMATO (REGEX) ===")
    print(relatorio_formato.to_string(index=False))
    return relatorio_formato

res_formato = diagnostico_conformidade_formato(df_recorte)

=== DIAGNÓSTICO DE CONFORMIDADE DE FORMATO (REGEX) ===
                                 Regra_Formato  Inconformidades  Percentual_Afetado (%)
            Cod_IBGE_Mun (7 dígitos numéricos)                0                     0.0
Nome_Municipio (sem espaços extras nas pontas)                0                     0.0
                Sigla_UF (2 letras maiúsculas)                0                     0.0



# 📊 Tabela Consolidada de Diagnóstico de Qualidade de Dados (S2iD 2021–2025)

| Dimensão | Problema Diagnosticado | % Afetado (Qtd.) | Flag / Coluna Criada | Ação Recomendada |
| :--- | :--- | :---: | :--- | :--- |
| **1. Completude** | **Sintática:** Nulos em colunas vitais | **0,00%** *(0)* | — | Nenhuma (estrutura 100% íntegra). |
| **1. Completude** | **Semântica:** "Falsos Zeros" (R\$ 0,00 em danos) | **36,10%** *(3.567)* | `flag_sem_declaracao_financeira` | Preservar dados; isolar os 36,1% em cálculos de custo médio. |
| **2. Consistência** | **Aritmética:** Divergência na soma de humanos | **6,13%** *(606)* | `DH_total_calculado`<br />`flag_inconsistencia_soma_humana` | Adotar `DH_total_calculado` como métrica oficial auditada. |
| **2. Consistência** | **Regra:** Status "Reconhecido" sem impacto | **1,79%** *(177)* | `flag_reconhecido_sem_impacto` | Filtrar os 177 registros em estudos de valoração de danos. |
| **3. Unicidade** | **Chave Primária:** Duplicidade no protocolo | **0,00%** *(0)* | `flag_duplicata_protocolo` | Integridade do banco mantida. |
| **3. Unicidade** | **Semântica:** Mesma ocorrência cadastrada 2x | **0,55%** *(54)* | `flag_duplicata_semantica` | Manter etiqueta; consolidar pelo maior dano/status antes de somar. |
| **4. Acurácia** | **Outliers Financeiros:** Prejuízos &gt; R\$ 5,31 mi | **9,57%** *(946)* | `flag_outlier_danos_materiais` | Usar mediana ou filtrar a flag para estimar o custo típico. |
| **4. Acurácia** | **Outliers Humanos:** Vítimas &gt; 280 pessoas | **14,72%** *(1.456)* | `flag_outlier_humanos` | Separar eventos locais de catástrofes de escala regional. |
| **5. Atualidade** | **Série Temporal:** Datas inválidas/fora do escopo | **0,00%** *(0)* | — | Série contínua e válida em todo o período (2021–2025). |
| **6. Conformidade** | **Padronização:** Erros de Regex (IBGE/UF) | **0,00%** *(0)* | — | Chave geográfica 100% pronta para cruzamentos GIS/IBGE. |


## Exportação do Dataset Auditado

**Descrição:** Exportação final da base de dados de desastres hidrológicos (2021–2025) contendo os dados brutos sanitizados e todas as **8 colunas derivadas de auditoria e flags booleanas** criadas durante o diagnóstico metodológico:
- **Colunas Criadas:**

1. \`flag\_sem\_declaracao\_financeira\` (Completude)
2. \`DH\_total\_calculado\` (Consistência Aritmética)
3. \`flag\_inconsistencia\_soma\_humana\` (Consistência Aritmética)
4. \`flag\_reconhecido\_sem\_impacto\` (Consistência de Regra)
5. \`flag\_duplicata\_protocolo\` (Unicidade)
6. \`flag\_duplicata\_semantica\` (Unicidade)
7. \`flag\_outlier\_danos\_materiais\` (Acurácia)
8. \`flag\_outlier\_humanos\` (Acurácia)

**Parâmetros de Saída:** Formato CSV, separador \`;\`, codificação \`UTF-8-SIG\` e preservação total das 9.890 ocorrências.

In [28]:
import pandas as pd
from google.colab import files

# 1. Definindo a lista exata de colunas que vão para o arquivo final
colunas_para_exportar = [
    # --- Colunas Vitais Base ---
    'Cod_IBGE_Mun',
    'Nome_Municipio',
    'Sigla_UF',
    'Data_Evento',
    'Status',
    'descricao_tipologia',
    'Protocolo_S2iD',
    'DM_total_danos_materiais',
    'PEPL_total_publico',

    # --- Novas Colunas Auditadas e Flags Criadas ---
    'DH_total_calculado',               # Substitui o total de danos humanos original (auditado e correto)
    'flag_sem_declaracao_financeira',    # Completude Semântica
    'flag_inconsistencia_soma_humana',   # Consistência Aritmética
    'flag_reconhecido_sem_impacto',      # Consistência de Regra
    'flag_duplicata_protocolo',         # Unicidade de Chave
    'flag_duplicata_semantica',         # Unicidade Semântica
    'flag_outlier_danos_materiais',     # Acurácia de Danos Materiais
    'flag_outlier_humanos'              # Acurácia de Afetados Humanos
]

# 2. Filtrando o DataFrame para conter SOMENTE as colunas selecionadas acima
df_exportar = df_recorte[colunas_para_exportar]

# 3. Exportando para CSV
nome_arquivo_csv = "S2iD_Desastres_Hidrologicos_Selecao_Auditada.csv"

df_exportar.to_csv(
    nome_arquivo_csv,
    index=False,
    encoding='utf-8-sig',
    sep=';'
)

print(f"✅ Arquivo enxuto '{nome_arquivo_csv}' gerado com sucesso!")
print(f"📊 Total de linhas: {len(df_exportar)} | Total de colunas filtradas: {len(df_exportar.columns)}")

# 4. Download automático no navegador
files.download(nome_arquivo_csv)

✅ Arquivo enxuto 'S2iD_Desastres_Hidrologicos_Selecao_Auditada.csv' gerado com sucesso!
📊 Total de linhas: 9890 | Total de colunas filtradas: 17


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>